# 04 Model Comparison

This notebook compares the standardized outputs from three models:

- U-Net
- DeepLabV3+
- SegFormer

## Objectives
- Load the standardized result files from each model
- Build a consolidated comparison table
- Compare the models on overall performance
- Compare precision and recall patterns
- Compare efficiency and predictive quality trade-offs

## Notes
This notebook assumes that each model folder contains:
- `metrics.csv`
- `training_history.csv`
- `config_used.yaml`

If some files are not available yet, placeholder checks are included so the notebook structure can still be built first.

In [ ]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to Python path
project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root))

# Import plotting utilities
from src.plot_utils import set_plot_style, save_figure, MODEL_COLORS, DEFAULT_FIGSIZE, WIDE_FIGSIZE

# Apply the global plotting style
set_plot_style()

print("Project root:", project_root)

## Define Standard Output Paths

This section defines the expected output locations for each model.
These paths follow the standardized project structure.

In [ ]:
# Define the expected model output directories
model_output_dirs = {
    "unet": project_root / "outputs" / "unet",
    "deeplabv3plus": project_root / "outputs" / "deeplabv3plus",
    "segformer": project_root / "outputs" / "segformer",
}

# Define the expected standardized files for each model
expected_files = {
    "metrics": "metrics.csv",
    "training_history": "training_history.csv",
    "config": "config_used.yaml",
}

model_output_dirs

## Check File Availability

Before loading the results, we first check whether the expected files exist for each model.
This allows us to build the notebook structure even if some outputs are still missing.

In [ ]:
# Check whether the standardized files exist for each model
file_check_records = []

for model_name, model_dir in model_output_dirs.items():
    record = {"model": model_name}

    for key, file_name in expected_files.items():
        file_path = model_dir / file_name
        record[f"{key}_exists"] = file_path.exists()
        record[f"{key}_path"] = str(file_path)

    file_check_records.append(record)

file_check_df = pd.DataFrame(file_check_records)
file_check_df

## Load Standardized Outputs

This section loads:
- `metrics.csv`
- `training_history.csv`
- `config_used.yaml`

The loading logic is designed to be robust to temporarily missing files.

In [ ]:
# Safe loading helpers
def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None

def safe_read_yaml(path):
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)
    return None


# Load outputs for all models
model_results = {}

for model_name, model_dir in model_output_dirs.items():
    model_results[model_name] = {
        "metrics": safe_read_csv(model_dir / expected_files["metrics"]),
        "training_history": safe_read_csv(model_dir / expected_files["training_history"]),
        "config": safe_read_yaml(model_dir / expected_files["config"]),
    }

print("Loaded model result keys:")
for model_name, result_dict in model_results.items():
    print(model_name, "->", list(result_dict.keys()))

In [ ]:
# Summarize loading status
loading_status = []

for model_name, result_dict in model_results.items():
    loading_status.append({
        "model": model_name,
        "metrics_loaded": result_dict["metrics"] is not None,
        "training_history_loaded": result_dict["training_history"] is not None,
        "config_loaded": result_dict["config"] is not None,
    })

loading_status_df = pd.DataFrame(loading_status)
loading_status_df

## Build the Consolidated Metrics Table

This table is the core summary for model comparison.
It will be used to compare overall predictive performance and efficiency.

In [ ]:
# Build a consolidated metrics table from all available model outputs
metrics_frames = []

for model_name, result_dict in model_results.items():
    metrics_df = result_dict["metrics"]

    if metrics_df is not None and not metrics_df.empty:
        metrics_frames.append(metrics_df.copy())

if len(metrics_frames) > 0:
    consolidated_metrics_df = pd.concat(metrics_frames, ignore_index=True)
else:
    consolidated_metrics_df = pd.DataFrame(columns=[
        "model", "iou", "f1", "precision", "recall", "params_m", "inference_time_ms"
    ])

consolidated_metrics_df

## Interpretation Placeholder: Overall Metrics

Write the final interpretation here after all model results are available.

Suggested points to discuss:
- Which model has the highest IoU and F1-score?
- Are the differences between models large or modest?
- Which model appears to provide the strongest overall performance?
- Does the ranking support the original expectation in the proposal?

## Compare Overall Performance

The first comparison focuses on the main segmentation quality metrics:
- IoU
- F1-score

In [ ]:
# Plot the main performance metrics if the consolidated table is available
if not consolidated_metrics_df.empty:
    metrics_to_plot = ["iou", "f1"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, metric in zip(axes, metrics_to_plot):
        model_names = consolidated_metrics_df["model"].tolist()
        values = consolidated_metrics_df[metric].tolist()

        colors = []
        for model_name in model_names:
            name_lower = str(model_name).lower()
            if "unet" in name_lower:
                colors.append(MODEL_COLORS["unet"])
            elif "deeplab" in name_lower:
                colors.append(MODEL_COLORS["deeplabv3plus"])
            elif "segformer" in name_lower:
                colors.append(MODEL_COLORS["segformer"])
            else:
                colors.append("#7f7f7f")

        ax.bar(model_names, values, color=colors)
        ax.set_title(f"{metric.upper()} Comparison")
        ax.set_ylabel(metric.upper())
        ax.set_xlabel("Model")

    plt.tight_layout()
    plt.show()
else:
    print("No consolidated metrics available yet.")

## Interpretation Placeholder: Main Performance Comparison

Write the final comparison here once the real values are available.

Suggested interpretation structure:
- The best-performing model on IoU is ...
- The best-performing model on F1-score is ...
- If the same model leads on both metrics, it suggests ...
- If the metrics disagree, discuss possible reasons.

## Compare Precision and Recall

Precision and Recall help us understand whether a model is more conservative or more aggressive in change prediction.
This prepares the ground for later false-positive and false-negative analysis.

In [ ]:
# Plot precision and recall comparison
if not consolidated_metrics_df.empty:
    metrics_to_plot = ["precision", "recall"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, metric in zip(axes, metrics_to_plot):
        model_names = consolidated_metrics_df["model"].tolist()
        values = consolidated_metrics_df[metric].tolist()

        colors = []
        for model_name in model_names:
            name_lower = str(model_name).lower()
            if "unet" in name_lower:
                colors.append(MODEL_COLORS["unet"])
            elif "deeplab" in name_lower:
                colors.append(MODEL_COLORS["deeplabv3plus"])
            elif "segformer" in name_lower:
                colors.append(MODEL_COLORS["segformer"])
            else:
                colors.append("#7f7f7f")

        ax.bar(model_names, values, color=colors)
        ax.set_title(f"{metric.capitalize()} Comparison")
        ax.set_ylabel(metric.capitalize())
        ax.set_xlabel("Model")

    plt.tight_layout()
    plt.show()
else:
    print("No consolidated metrics available yet.")

## Interpretation Placeholder: Precision and Recall

Write the final interpretation here after the actual results are loaded.

Suggested points to discuss:
- Which model has the highest precision?
- Which model has the highest recall?
- Does any model appear more conservative or more aggressive?
- What might this imply for false positives and false negatives?

## Compare Efficiency and Performance Trade-off

This section compares model quality and computational cost together.
It helps us discuss practical deployment trade-offs.

In [ ]:
# Plot efficiency vs performance trade-off
if not consolidated_metrics_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))

    for _, row in consolidated_metrics_df.iterrows():
        model_name = row["model"]
        f1_value = row["f1"]
        time_value = row["inference_time_ms"]

        name_lower = str(model_name).lower()
        if "unet" in name_lower:
            color = MODEL_COLORS["unet"]
        elif "deeplab" in name_lower:
            color = MODEL_COLORS["deeplabv3plus"]
        elif "segformer" in name_lower:
            color = MODEL_COLORS["segformer"]
        else:
            color = "#7f7f7f"

        ax.scatter(time_value, f1_value, s=120, color=color)
        ax.text(time_value, f1_value, str(model_name), fontsize=10)

    ax.set_title("Efficiency vs Performance")
    ax.set_xlabel("Inference Time (ms)")
    ax.set_ylabel("F1-score")

    plt.tight_layout()
    plt.show()
else:
    print("No consolidated metrics available yet.")

## Interpretation Placeholder: Efficiency and Deployment Trade-off

Write the final interpretation here after the actual results are loaded.

Suggested points to discuss:
- Which model provides the best F1-score?
- Which model is the fastest?
- Does the most accurate model also have the highest computational cost?
- Which model seems most suitable for large-scale automatic screening?
- Which model seems more suitable for high-precision manual review support?

## Optional: Compare Training Stability

This section can be used to compare convergence patterns across models if all `training_history.csv` files are available.

In [ ]:
# Optional: compare training history if available
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

history_available = False

for model_name, result_dict in model_results.items():
    history_df = result_dict["training_history"]

    if history_df is not None and not history_df.empty:
        history_available = True

        name_lower = str(model_name).lower()
        if "unet" in name_lower:
            color = MODEL_COLORS["unet"]
        elif "deeplab" in name_lower:
            color = MODEL_COLORS["deeplabv3plus"]
        elif "segformer" in name_lower:
            color = MODEL_COLORS["segformer"]
        else:
            color = "#7f7f7f"

        axes[0].plot(history_df["epoch"], history_df["train_loss"], label=model_name, color=color)
        axes[1].plot(history_df["epoch"], history_df["val_loss"], label=model_name, color=color)

axes[0].set_title("Training Loss Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train Loss")
axes[0].legend()

axes[1].set_title("Validation Loss Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Loss")
axes[1].legend()

plt.tight_layout()

if history_available:
    plt.show()
else:
    plt.close(fig)
    print("Training history files are not fully available yet.")

## Key Findings Placeholder

Write the final summary here after all results are available.

Suggested summary structure:
1. Overall, the strongest model in terms of segmentation quality is ...
2. In terms of precision-recall balance, ...
3. In terms of efficiency, ...
4. The model that seems most suitable for large-scale monitoring is ...
5. The model that seems most suitable for high-precision support is ...

## Export Placeholder

This section can later be used to:
- save the consolidated results table
- export the main figures for the final report
- export presentation-ready figure versions

In [ ]:
# Placeholder for future exports
# Example:
# consolidated_metrics_df.to_csv(project_root / "outputs" / "tables" / "model_comparison_table.csv", index=False)

print("Export section placeholder ready.")